## Systematic Ablation Study for Hybrid MST-Conformer Architecture (v5 — Ablation v2)

In [ ]:
# ==== BLOCK 1: SETUP & DEPENDENCIES ====
from google.colab import drive
drive.mount('/content/drive')
import time, os, re, zipfile, fnmatch, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as sig
from scipy import stats as sp_stats
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, roc_curve, auc, confusion_matrix, classification_report

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q scipy scikit-learn matplotlib pandas tqdm optuna wandb seaborn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import optuna
from optuna.exceptions import TrialPruned
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Output & Precomputed Directory
PRECOMPUTED_DIR = '/content/drive/MyDrive/UPVEHU/Investigacion/articulo_panama_2026/Ablation'
os.makedirs(PRECOMPUTED_DIR, exist_ok=True)

print("Dependencies loaded successfully.")
print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
print(f"Target Save Directory: {PRECOMPUTED_DIR}")

In [ ]:
###### BLOCK 2A: LOAD MATLAB TABLES   ######

from google.colab import drive
drive.mount('/content/drive')

import h5py
import numpy as np

# ============================================================
# FEATURES FILE
# ============================================================

import pandas as pd

df_feat = pd.read_csv('/content/drive/MyDrive/UPVEHU/Investigacion/articulo_panama_2026/Ablation/All_ECR_9.csv')

print(df_feat.head())
print(df_feat.shape)
print(df_feat.columns)
print(df_feat.columns.tolist())


print("="*60)
print("FEATURES FILE")
print("="*60)

print("Keys:")
print(list(df_feat.keys()))

for k in df_feat.keys():
    try:
        print(f"{k}: shape={df_feat[k].shape}, dtype={df_feat[k].dtype}")
    except:
        print(k)

# ============================================================
# SIGNALS FILE
# ============================================================

signals_mat = h5py.File('/content/drive/MyDrive/UPVEHU/Investigacion/articulo_panama_2026/Ablation/PER_SUBEJCT_WESAD_raw_signals_norm_allSubjects.mat','r')

print("\n")
print("="*60)
print("SIGNALS FILE")
print("="*60)

print("Keys:")
print(list(signals_mat.keys()))

for k in signals_mat.keys():
    try:
        print(f"{k}: shape={signals_mat[k].shape}, dtype={signals_mat[k].dtype}")
    except:
        print(k)

# ============================================================
# DETAILS OF ALL_ECR
# ============================================================

if 'All_ECR' in df_feat:

    print("\n")
    print("="*60)
    print("DETAILS OF All_ECR")
    print("="*60)

    print(type(df_feat['All_ECR']))
    print(df_feat['All_ECR'])

# ============================================================
# DETAILS OF SUBJECT
# ============================================================

if 'Subject' in signals_mat:

    print("\n")
    print("="*60)
    print("DETAILS OF Subject")
    print("="*60)

    print(signals_mat['Subject'])
    print("shape =", signals_mat['Subject'].shape)
    print("dtype =", signals_mat['Subject'].dtype)

    try:
        print("raw values:")
        print(np.array(signals_mat['Subject']))
    except Exception as e:
        print(e)

# ============================================================
# DETAILS OF X_raw_norm
# ============================================================

if 'X_raw_norm' in signals_mat:

    print("\n")
    print("="*60)
    print("DETAILS OF X_raw_norm")
    print("="*60)

    Xtmp = np.array(signals_mat['X_raw_norm'])

    print("shape =", Xtmp.shape)
    print("dtype =", Xtmp.dtype)

# ============================================================
# DETAILS OF y
# ============================================================

if 'y' in signals_mat:

    print("\n")
    print("="*60)
    print("DETAILS OF y")
    print("="*60)

    ytmp = np.array(signals_mat['y'])

    print("shape =", ytmp.shape)
    print("dtype =", ytmp.dtype)

    try:
        print("unique values:", np.unique(ytmp))
    except:
        pass


In [ ]:
###### BLOCK 2B: DELETE USELESS FEATURES   ######

import numpy as np

X_raw_norm = np.array(signals_mat['X_raw_norm'])
X_raw_norm = np.transpose(X_raw_norm, (2,1,0))

print(X_raw_norm.shape)

print(df_feat.shape)

FEATURE_NAMES = [
    c for c in df_feat.columns
    if c not in [
        'Subject',
        'y',
        'CC_q1',
        'CC_q2',
        'CC_q3',
        'CC_q4'
    ]
]

N_FEATURES = len(FEATURE_NAMES)

print(N_FEATURES)
print(FEATURE_NAMES)

X_raw_norm = np.array(signals_mat['X_raw_norm'])

X_raw_norm = np.transpose(
    X_raw_norm,
    (2,1,0)
)

print(X_raw_norm.shape)

In [ ]:
###### BLOCK 3A: CONDITION SIGNALS FOR USING IN PYTHON   ######

SUBJECTS = ['S2', 'S3', 'S4', 'S5', 'S6','S7', 'S8', 'S9', 'S10', 'S11',
            'S13', 'S14', 'S15', 'S16', 'S17']
subject_data = {}
subject_data_norm = {}

for sid in SUBJECTS:
    sid_sync = f"{sid}_sync"
    idx = (df_feat['Subject'] == sid_sync)
    X_feat_sid = (df_feat.loc[idx, FEATURE_NAMES].values.astype(np.float32))

    y_sid = (df_feat.loc[idx, 'y'].values.astype(np.int64))

    X_raw_sid = X_raw_norm[idx.values]

    subject_data[sid] = (X_raw_sid,X_feat_sid,y_sid);
    subject_data_norm[sid] = (X_raw_sid,X_feat_sid,y_sid)

    print(
        f"{sid}: "
        f"{len(y_sid)} windows | "
        f"X_raw={X_raw_sid.shape} | "
        f"X_feat={X_feat_sid.shape}"
    )
print(subject_data_norm.keys())

xr, xf, y = subject_data_norm['S2']

print(xr.shape)
print(xf.shape)
print(y.shape)
print("N_FEATURES =", xf.shape[1])

# ==== GLOBAL CONFIG ====

FS = 700

WINDOW_SEC = 20
WINDOW_SIZE = WINDOW_SEC * FS

STRIDE = WINDOW_SIZE // 4

NUM_CHANNELS = 2

LABEL_MAP = {
    1: 0,
    2: 1,
    3: 1,
    4: 0
}

RANDOM_STATE = 42

In [ ]:
###### BLOCK 3B: CHECKING DATASET IS OKAY   ######

total_windows = 0

for sid in SUBJECTS:
    total_windows += len(subject_data_norm[sid][2])

print(total_windows)

print('Subject Class0 Class1')
for sid in SUBJECTS:
    y = subject_data_norm[sid][2]

    print(
        sid,
        np.sum(y == 0),
        np.sum(y == 1)
    )

print("N_FEATURES =", N_FEATURES)

for sid in SUBJECTS:
    xr, xf, y = subject_data_norm[sid]

    assert xr.shape[2] == 2
    assert xf.shape[1] == N_FEATURES

print("Dataset check passed.")

In [ ]:
# ==== BLOCK 4: PYTORCH DATASET & LOSO CROSS-VALIDATION SPLITS ====
class WESADDatasetV4(Dataset):
    '''Dataset returning (raw_signal, features, label) with optional data augmentation.'''
    def __init__(self, X_raw, X_feat, y, augment=False):
        self.X_raw = torch.FloatTensor(X_raw).permute(0, 2, 1)  # (N, C, T)
        self.X_feat = torch.FloatTensor(X_feat)                  # (N, F)
        self.y = torch.FloatTensor(y)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x_raw = self.X_raw[idx].clone()
        x_feat = self.X_feat[idx].clone()
        if self.augment:
            x_raw = self._augment_raw(x_raw)
            x_feat = self._augment_feat(x_feat)
        return x_raw, x_feat, self.y[idx]

    def _augment_raw(self, x):
        C, T = x.shape
        if torch.rand(1).item() < 0.7:
            x = x + torch.randn_like(x) * 0.02
        if torch.rand(1).item() < 0.7:
            scale = 0.8 + torch.rand(1).item() * 0.4
            x = x * scale
        if torch.rand(1).item() < 0.5:
            shift = int(torch.randint(-T // 10, T // 10 + 1, (1,)).item())
            x = torch.roll(x, shifts=shift, dims=-1)
        if torch.rand(1).item() < 0.5:
            mask_len = int(torch.randint(T // 20, T // 5, (1,)).item())
            mask_start = int(torch.randint(0, T - mask_len, (1,)).item())
            x[:, mask_start:mask_start + mask_len] = 0.0
        if torch.rand(1).item() < 0.2:
            ch = int(torch.randint(0, C, (1,)).item())
            x[ch, :] = 0.0
        return x

    def _augment_feat(self, f):
        if torch.rand(1).item() < 0.5:
            f = f + torch.randn_like(f) * 0.05
        return f

def get_loso_splits_v4(subject_data_norm, test_sid, val_sid=None): # Version modificada por Asier para garantizar rotacion
    '''
    Builds Train / Validation / Test partitions.

    If val_sid is provided, it is used directly.
    Otherwise the validation subject is chosen as the
    next subject in SUBJECTS (circular rotation).
    '''

    sids = SUBJECTS.copy()

    if val_sid is None:
        test_idx = sids.index(test_sid)
        val_sid = sids[(test_idx + 1) % len(sids)]

    # Test
    ts_Xr, ts_Xf, ts_y = subject_data_norm[test_sid]

    # Validation
    vl_Xr, vl_Xf, vl_y = subject_data_norm[val_sid]

    # Train = all except test and validation
    train_sids = [
        s for s in sids
        if s != test_sid and s != val_sid
    ]

    tr_Xr = np.vstack([
        subject_data_norm[s][0]
        for s in train_sids
    ])

    tr_Xf = np.vstack([
        subject_data_norm[s][1]
        for s in train_sids
    ])

    tr_y = np.concatenate([
        subject_data_norm[s][2]
        for s in train_sids
    ])

    return (
        tr_Xr, tr_Xf, tr_y,
        vl_Xr, vl_Xf, vl_y,
        ts_Xr, ts_Xf, ts_y,
        val_sid
    )
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Computing Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')

In [ ]:
# ==== BLOCK 5: MODULAR ARCHITECTURE & ALL 5 ABLATION VARIANTS ====

# ----------------------------------------------------
# 1. CORE BUILDING BLOCKS
# ----------------------------------------------------
class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation=1):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation)
    def forward(self, x):
        if self.pad > 0: x = F.pad(x, (self.pad, 0))
        return self.conv(x)

class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(in_ch, out_ch, kernel_size, dilation),
            nn.BatchNorm1d(out_ch), nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(out_ch, out_ch, kernel_size, dilation),
            nn.BatchNorm1d(out_ch), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.net(x) + self.residual(x))

class MultiScaleTCN(nn.Module):
    '''Proposed Multi-Scale TCN with 3 parallel dilation branches: small, medium, large.'''
    def __init__(self, in_ch, filters, ks, dropout, **kwargs):
        super().__init__()
        self.branch_s = nn.Sequential(
            TCNBlock(in_ch, filters, ks, 1, dropout),
            TCNBlock(filters, filters, ks, 2, dropout),
            TCNBlock(filters, filters, ks, 4, dropout),
        )
        self.branch_m = nn.Sequential(
            TCNBlock(in_ch, filters, ks, 1, dropout),
            TCNBlock(filters, filters, ks, 4, dropout),
            TCNBlock(filters, filters, ks, 16, dropout),
        )
        self.branch_l = nn.Sequential(
            TCNBlock(in_ch, filters, ks, 1, dropout),
            TCNBlock(filters, filters, ks, 8, dropout),
            TCNBlock(filters, filters, ks, 64, dropout),
        )
    def forward(self, x):
        return torch.cat([self.branch_s(x), self.branch_m(x), self.branch_l(x)], dim=1)

class SingleDilationTCN(nn.Module):
    '''Ablation Baseline: Single-Dilation TCN (dilation=1 for all blocks, no multiscale rates).'''
    def __init__(self, in_ch, out_ch, ks, dropout, **kwargs):
        super().__init__()
        self.net = nn.Sequential(
            TCNBlock(in_ch, out_ch, ks, 1, dropout),
            TCNBlock(out_ch, out_ch, ks, 1, dropout),
            TCNBlock(out_ch, out_ch, ks, 1, dropout),
        )
    def forward(self, x):
        return self.net(x)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(pos * div[:-1])
        else:
            pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class FeatureMLP(nn.Module):
    '''Feature processing MLP for 9 handcrafted features.'''
    def __init__(self, n_features=9, feat_hidden=32, dropout=0.4, **kwargs):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, feat_hidden),
            nn.BatchNorm1d(feat_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(feat_hidden, feat_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

# ----------------------------------------------------
# 2. PROPOSED FULL MODEL & ABLATION MODEL CLASSES
# ----------------------------------------------------

# [PROPOSED FULL MODEL]
class HybridMSTConformer(nn.Module):
    '''Proposed Full Model: Multi-Scale TCN + Transformer Encoder + Feature MLP Branch.'''
    def __init__(self, in_channels=2, tcn_filters=32, kernel_size=3,
                 n_layers=2, n_heads=2, d_ff=128, seq_len_out=64,
                 n_features=9, feat_hidden=32, dropout=0.4, **kwargs):
        super().__init__()
        d_model = tcn_filters * 3
        self.tcn = MultiScaleTCN(in_channels, tcn_filters, kernel_size, dropout)
        self.pool = nn.AdaptiveAvgPool1d(seq_len_out)
        self.norm_in = nn.LayerNorm(d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=seq_len_out)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.feat_branch = FeatureMLP(n_features, feat_hidden, dropout)
        self.head = nn.Sequential(
            nn.Linear(d_model + feat_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x_raw, x_feat=None):
        x = self.tcn(x_raw)
        x = self.pool(x)
        x = x.permute(0, 2, 1)
        x = self.norm_in(x)
        x = self.pos_enc(x)
        x = self.transformer(x)
        x_raw_out = x.mean(dim=1)
        x_feat_out = self.feat_branch(x_feat)
        fused = torch.cat([x_raw_out, x_feat_out], dim=1)
        return self.head(fused).squeeze(-1)

# [VARIANT (i) — WITHOUT FEATURE BRANCH]
class AblationNoFeatures(nn.Module):
    '''Variant (i): Multi-Scale TCN + Transformer Encoder only (no handcrafted feature branch).'''
    def __init__(self, in_channels=2, tcn_filters=32, kernel_size=3,
                 n_layers=2, n_heads=2, d_ff=128, seq_len_out=64,
                 dropout=0.4, **kwargs):
        super().__init__()
        d_model = tcn_filters * 3
        self.tcn = MultiScaleTCN(in_channels, tcn_filters, kernel_size, dropout)
        self.pool = nn.AdaptiveAvgPool1d(seq_len_out)
        self.norm_in = nn.LayerNorm(d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=seq_len_out)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x_raw, x_feat=None):
        x = self.tcn(x_raw)
        x = self.pool(x)
        x = x.permute(0, 2, 1)
        x = self.norm_in(x)
        x = self.pos_enc(x)
        x = self.transformer(x)
        x_raw_out = x.mean(dim=1)
        return self.head(x_raw_out).squeeze(-1)

# [VARIANT (ii) — FEATURE BRANCH ALONE]
class AblationFeatureOnly(nn.Module):
    '''Variant (ii): 9 Handcrafted features with MLP only (no raw signal branch).'''
    def __init__(self, n_features=9, feat_hidden=32, dropout=0.4, **kwargs):
        super().__init__()
        self.feat_branch = FeatureMLP(n_features, feat_hidden, dropout)
        self.head = nn.Sequential(
            nn.Linear(feat_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x_raw=None, x_feat=None):
        x_feat_out = self.feat_branch(x_feat)
        return self.head(x_feat_out).squeeze(-1)

# [VARIANT (iii) — WITHOUT TCN]
class AblationNoTCN(nn.Module):
    '''Variant (iii): Transformer branch + Feature branch (without Multi-Scale TCN).'''
    def __init__(self, in_channels=2, tcn_filters=32,
                 n_layers=2, n_heads=2, d_ff=128, seq_len_out=64,
                 n_features=9, feat_hidden=32, dropout=0.4, **kwargs):
        super().__init__()
        d_model = tcn_filters * 3
        self.proj = nn.Sequential(
            nn.Conv1d(in_channels, d_model, kernel_size=1),
            nn.BatchNorm1d(d_model),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(seq_len_out)
        self.norm_in = nn.LayerNorm(d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=seq_len_out)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.feat_branch = FeatureMLP(n_features, feat_hidden, dropout)
        self.head = nn.Sequential(
            nn.Linear(d_model + feat_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x_raw, x_feat=None):
        x = self.proj(x_raw)
        x = self.pool(x)
        x = x.permute(0, 2, 1)
        x = self.norm_in(x)
        x = self.pos_enc(x)
        x = self.transformer(x)
        x_raw_out = x.mean(dim=1)
        x_feat_out = self.feat_branch(x_feat)
        fused = torch.cat([x_raw_out, x_feat_out], dim=1)
        return self.head(fused).squeeze(-1)

# [VARIANT (iv) — WITHOUT TRANSFORMER]
class AblationNoTransformer(nn.Module):
    '''Variant (iv): Multi-Scale TCN branch + Feature branch (without Transformer Encoder).'''
    def __init__(self, in_channels=2, tcn_filters=32, kernel_size=3,
                 n_features=9, feat_hidden=32, dropout=0.4, **kwargs):
        super().__init__()
        d_model = tcn_filters * 3
        self.tcn = MultiScaleTCN(in_channels, tcn_filters, kernel_size, dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.feat_branch = FeatureMLP(n_features, feat_hidden, dropout)
        self.head = nn.Sequential(
            nn.Linear(d_model + feat_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x_raw, x_feat=None):
        x = self.tcn(x_raw)
        x_raw_out = self.pool(x).squeeze(-1)
        x_feat_out = self.feat_branch(x_feat)
        fused = torch.cat([x_raw_out, x_feat_out], dim=1)
        return self.head(fused).squeeze(-1)

# [VARIANT (v) — SINGLE-DILATION TCN]
class AblationSingleDilationTCN(nn.Module):
    '''Variant (v): Single-Dilation TCN (dilation=1) + Transformer Encoder + Feature branch.'''
    def __init__(self, in_channels=2, tcn_filters=32, kernel_size=3,
                 n_layers=2, n_heads=2, d_ff=128, seq_len_out=64,
                 n_features=9, feat_hidden=32, dropout=0.4, **kwargs):
        super().__init__()
        d_model = tcn_filters * 3
        self.tcn = SingleDilationTCN(in_channels, d_model, kernel_size, dropout)
        self.pool = nn.AdaptiveAvgPool1d(seq_len_out)
        self.norm_in = nn.LayerNorm(d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=seq_len_out)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.feat_branch = FeatureMLP(n_features, feat_hidden, dropout)
        self.head = nn.Sequential(
            nn.Linear(d_model + feat_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x_raw, x_feat=None):
        x = self.tcn(x_raw)
        x = self.pool(x)
        x = x.permute(0, 2, 1)
        x = self.norm_in(x)
        x = self.pos_enc(x)
        x = self.transformer(x)
        x_raw_out = x.mean(dim=1)
        x_feat_out = self.feat_branch(x_feat)
        fused = torch.cat([x_raw_out, x_feat_out], dim=1)
        return self.head(fused).squeeze(-1)

# ----------------------------------------------------
# 3. UNIFIED FACTORY & SANITY CHECKS
# ----------------------------------------------------
ABLATION_MODELS = {
    'full': HybridMSTConformer,
    'no_features': AblationNoFeatures,
    'feature_only': AblationFeatureOnly,
    'no_tcn': AblationNoTCN,
    'no_transformer': AblationNoTransformer,
    'single_dilation_tcn': AblationSingleDilationTCN,
}

def build_ablation_model(variant_name, in_channels=2, n_features=9, **kwargs):
    '''Unified factory function to instantiate any ablation model safely passing kwargs.'''
    if variant_name not in ABLATION_MODELS:
        raise ValueError(f"Unknown variant '{variant_name}'. Supported: {list(ABLATION_MODELS.keys())}")
    return ABLATION_MODELS[variant_name](in_channels=in_channels, n_features=n_features, **kwargs)

# Sanity Check for All 6 Models
print("="*75)
print(f"{'Variant Key':<22s} | {'Model Class':<26s} | {'Params':>10s} | {'Output':>8s}")
print("="*75)
_xr = torch.randn(4, NUM_CHANNELS, WINDOW_SIZE)
_xf = torch.randn(4, N_FEATURES)
for var_key, model_cls in ABLATION_MODELS.items():
    _m = build_ablation_model(var_key, in_channels=NUM_CHANNELS, n_features=N_FEATURES)
    _out = _m(_xr, _xf)
    _p = sum(p.numel() for p in _m.parameters())
    print(f"{var_key:<22s} | {model_cls.__name__:<26s} | {_p:>10,d} | {str(list(_out.shape)):>8s}")
    del _m
del _xr, _xf
print("="*75)
print("All 6 model variants verified successfully!")

In [ ]:
# ==== BLOCK 6: FOCAL LOSS, OPTIMAL THRESHOLDING & TRAINING ENGINE ====

class FocalLoss(nn.Module):
    '''Focal Loss with class balance alpha and label smoothing.'''
    def __init__(self, alpha=0.5, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ls = label_smoothing

    def forward(self, logits, targets):
        if self.ls > 0:
            targets = targets * (1 - self.ls) + 0.5 * self.ls
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.sigmoid(logits)
        pt = targets * pt + (1 - targets) * (1 - pt)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        return (alpha_t * ((1 - pt) ** self.gamma) * bce).mean()

def find_optimal_threshold(probs, labels):
    '''Grid search on validation set to find threshold maximizing Balanced Accuracy.'''
    best_t, best_score = 0.5, -1.0
    for t in np.arange(0.05, 0.95, 0.01):
        preds = (probs >= t).astype(int)
        labs = labels.astype(int)
        tp = np.sum((preds == 1) & (labs == 1))
        tn = np.sum((preds == 0) & (labs == 0))
        fp = np.sum((preds == 1) & (labs == 0))
        fn = np.sum((preds == 0) & (labs == 1))
        bal_acc = ((tp / max(tp + fn, 1)) + (tn / max(tn + fp, 1))) / 2.0
        if bal_acc > best_score:
            best_score, best_t = bal_acc, t
    return best_t, best_score

def train_one_epoch_ablation(model, loader, optimizer, criterion, scheduler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xr, xf, yb in loader:
        xr, xf, yb = xr.to(device), xf.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xr, xf)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler: scheduler.step()
        total_loss += loss.item() * len(yb)
        correct += ((torch.sigmoid(logits) >= 0.5).long() == yb.long()).sum().item()
        total += len(yb)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate_ablation(model, loader, criterion):
    model.eval()
    total_loss, probs_list, labels_list = 0.0, [], []
    for xr, xf, yb in loader:
        xr, xf, yb = xr.to(device), xf.to(device), yb.to(device)
        logits = model(xr, xf)
        total_loss += criterion(logits, yb).item() * len(yb)
        probs_list.append(torch.sigmoid(logits).cpu().numpy())
        labels_list.append(yb.cpu().numpy())
    probs = np.concatenate(probs_list)
    labels = np.concatenate(labels_list)
    preds = (probs >= 0.5).astype(int)
    has_both = len(np.unique(labels)) > 1
    return {
        'loss': total_loss / len(labels),
        'acc': accuracy_score(labels, preds),
        'bal_acc': balanced_accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, zero_division=0),
        'auc': roc_auc_score(labels, probs) if has_both else 0.0,
        'probs': probs, 'labels': labels
    }

def train_ablation_model(model, train_loader, val_loader, cfg, verbose=False):
    '''Standardized training loop with OneCycleLR scheduler and early stopping on validation F1.'''
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                  weight_decay=cfg.get('wd', 5e-4))
    train_labels = train_loader.dataset.y.numpy()
    pos_ratio = train_labels.mean()
    alpha = 1.0 - pos_ratio
    criterion = FocalLoss(alpha=alpha, gamma=cfg.get('focal_gamma', 2.0),
                          label_smoothing=cfg.get('label_smoothing', 0.05))
    total_steps = cfg['epochs'] * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg['lr'], total_steps=total_steps
    )

    best_f1, best_state, patience_cnt = -1.0, None, 0
    for epoch in range(cfg['epochs']):
        t_loss, t_acc = train_one_epoch_ablation(model, train_loader, optimizer, criterion, scheduler)
        v = evaluate_ablation(model, val_loader, criterion)
        if v['f1'] > best_f1:
            best_f1 = v['f1']
            best_state = {k: val.cpu().clone() for k, val in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if verbose and (epoch + 1) % 10 == 0:
            print(f'    Ep {epoch+1:02d}/{cfg["epochs"]} | TrLoss={t_loss:.4f} TrAcc={t_acc:.3f} | '
                  f'VlF1={v["f1"]:.3f} VlBalAcc={v["bal_acc"]:.3f} VlAUC={v["auc"]:.3f}')
        if patience_cnt >= cfg.get('patience', 8):
            if verbose: print(f'    Early stop triggered at epoch {epoch+1}')
            break
    if best_state:
        model.load_state_dict(best_state)
    return model, best_f1

print("Standardized training and evaluation engine ready.")

Standardized training and evaluation engine ready.


In [ ]:
# ==== BLOCK 7: STANDARDIZED OPTUNA ENGINE & PRE-OPTIMIZED HYPERPARAMETER MANAGER ====

search_sids = list(subject_data_norm.keys())[:6]
search_data = {s: subject_data_norm[s] for s in search_sids}
print(f'Optuna search subjects ({len(search_sids)}): {search_sids}')

# Pre-optimized Hyperparameter Database from 25-Trial Optuna Searches
# Bypasses hours of tuning while retaining exact experimental equivalence
PREV_OPTIMIZED_PARAMS = {
    'full': {
        'lr': 0.0004375784017644402,
        'dropout': 0.33656779957923044,
        'focal_gamma': 1.3558805007524795,
        'label_smoothing': 0.011583696354403306,
        'wd': 0.00026867742118364714,
        'batch_size': 64,
        'tcn_filters': 16,
        'n_layers': 1,
        'n_heads': 2,
        'd_ff': 128,
        'seq_len_out': 64,
        #'feat_hidden': 16,
        'feat_hidden': 32,
        'epochs': 50,
        'patience': 10
    },
    'no_features': {
        'lr': 0.0004107511959323418,
        'dropout': 0.5501366565106951,
        'focal_gamma': 1.650031882498103,
        'label_smoothing': 0.027479282848133557,
        'wd': 0.0012494055137952551,
        'batch_size': 32,
        'tcn_filters': 16,
        'n_layers': 1,
        'n_heads': 2,
        'd_ff': 64,
        'seq_len_out': 32,
        'epochs': 50,
        'patience': 10
    },
    'feature_only': {
        'lr': 0.0010,
        'dropout': 0.35,
        'focal_gamma': 1.50,
        'label_smoothing': 0.02,
        'wd': 0.001,
        'batch_size': 64,
        'feat_hidden': 32,
        'epochs': 50,
        'patience': 10
    },
    'no_tcn': {
        'lr': 0.00035,
        'dropout': 0.35,
        'focal_gamma': 1.50,
        'label_smoothing': 0.02,
        'wd': 0.0005,
        'batch_size': 64,
        'tcn_filters': 16,
        'n_layers': 1,
        'n_heads': 2,
        'd_ff': 128,
        'seq_len_out': 64,
        #'feat_hidden': 16,
        'feat_hidden': 32,
        'epochs': 50,
        'patience': 10
    },
    'no_transformer': {
        'lr': 0.00040,
        'dropout': 0.40,
        'focal_gamma': 1.50,
        'label_smoothing': 0.02,
        'wd': 0.0005,
        'batch_size': 64,
        'tcn_filters': 32,
        'feat_hidden': 32,
        'epochs': 50,
        'patience': 10
    },
    'single_dilation_tcn': {
        'lr': 0.00040,
        'dropout': 0.35,
        'focal_gamma': 1.50,
        'label_smoothing': 0.02,
        'wd': 0.0005,
        'batch_size': 64,
        'tcn_filters': 16,
        'n_layers': 1,
        'n_heads': 2,
        'd_ff': 128,
        'seq_len_out': 64,
        #'feat_hidden': 16,
        'feat_hidden': 32,
        'epochs': 50,
        'patience': 10
    }
}

def optuna_objective_ablation(trial, variant_name):
    '''Standardized Optuna objective function with identical search budget across all variants.'''
    cfg = {
        'lr':               trial.suggest_float('lr', 5e-5, 2e-3, log=True),
        'dropout':          trial.suggest_float('dropout', 0.25, 0.6),
        'focal_gamma':      trial.suggest_float('focal_gamma', 1.0, 3.0),
        'label_smoothing':  trial.suggest_float('label_smoothing', 0.0, 0.1),
        'wd':               trial.suggest_float('wd', 1e-4, 1e-2, log=True),
        'batch_size':       trial.suggest_categorical('batch_size', [32, 64]),
        'epochs': 30,
        'patience': 6,
    }

    if variant_name != 'feature_only':
        cfg['tcn_filters'] = trial.suggest_categorical('tcn_filters', [16, 32])
    else:
        cfg['tcn_filters'] = 32

    if variant_name not in ['no_transformer', 'feature_only']:
        cfg['n_layers']    = trial.suggest_int('n_layers', 1, 2)
        cfg['n_heads']     = trial.suggest_categorical('n_heads', [2, 4])
        cfg['d_ff']        = trial.suggest_categorical('d_ff', [64, 128])
        cfg['seq_len_out'] = trial.suggest_categorical('seq_len_out', [32, 64])
        d_model = cfg['tcn_filters'] * 3
        if d_model % cfg['n_heads'] != 0:
            raise TrialPruned()

    if variant_name != 'no_features':
        cfg['feat_hidden'] = trial.suggest_categorical('feat_hidden', [16, 32, 64])

    fold_f1s = []
    for i, test_sid in enumerate(search_sids[:3]):
        val_sid = search_sids[(i + 1) % len(search_sids)]
        splits = get_loso_splits_v4(search_data, test_sid, val_sid)
        trXr, trXf, trY, vlXr, vlXf, vlY, tsXr, tsXf, tsY, _ = splits

        tr_ds = WESADDatasetV4(trXr, trXf, trY, augment=True)
        vl_ds = WESADDatasetV4(vlXr, vlXf, vlY)
        ts_ds = WESADDatasetV4(tsXr, tsXf, tsY)
        tr_loader = DataLoader(tr_ds, batch_size=cfg['batch_size'], shuffle=True, drop_last=True)
        vl_loader = DataLoader(vl_ds, batch_size=cfg['batch_size'])
        ts_loader = DataLoader(ts_ds, batch_size=cfg['batch_size'])

        model = build_ablation_model(
            variant_name, in_channels=NUM_CHANNELS, n_features=N_FEATURES, **cfg
        ).to(device)

        model, _ = train_ablation_model(model, tr_loader, vl_loader, cfg, verbose=False)
        crit = FocalLoss(gamma=cfg['focal_gamma'], label_smoothing=cfg['label_smoothing'])
        vl_m = evaluate_ablation(model, vl_loader, crit)
        opt_t, _ = find_optimal_threshold(vl_m['probs'], vl_m['labels'])
        ts_m = evaluate_ablation(model, ts_loader, crit)
        ts_preds = (ts_m['probs'] >= opt_t).astype(int)
        fold_f1 = f1_score(ts_m['labels'], ts_preds, zero_division=0)
        fold_f1s.append(fold_f1)

        del model; torch.cuda.empty_cache()
        trial.report(np.mean(fold_f1s), i)
        if trial.should_prune():
            raise TrialPruned()

    return float(np.mean(fold_f1s))

def get_variant_hyperparameters(variant_name, use_previous=True, run_optuna=False, n_trials=25):
    '''Returns hyperparameter dictionary. Can load saved JSON, use pre-optimized database, or run Optuna.'''
    cfg_file = os.path.join(PRECOMPUTED_DIR, f'best_config_{variant_name}.json')

    # 1. Check if saved file already exists on Drive
    if os.path.isfile(cfg_file) and not run_optuna:
        try:
            with open(cfg_file, 'r') as f:
                cfg = json.load(f)
            print(f"[HYPERPARAMS] Loaded saved config from {os.path.basename(cfg_file)}")
            return cfg
        except Exception as e:
            print(f"[HYPERPARAMS] Warning reading {cfg_file}: {e}")

    # 2. Use previous pre-optimized configuration
    if use_previous and not run_optuna:
        cfg = PREV_OPTIMIZED_PARAMS.get(variant_name, PREV_OPTIMIZED_PARAMS['full'])
        print(f"[HYPERPARAMS] Using previous pre-optimized hyperparameters for '{variant_name}' (bypassing Optuna search).")
        # Save to PRECOMPUTED_DIR
        with open(cfg_file, 'w') as f:
            json.dump(cfg, f, indent=2)
        return cfg

    # 3. Otherwise run Optuna search
    print(f"\n{'='*70}\n[OPTUNA] Starting Search for Variant: {variant_name} ({n_trials} trials)\n{'='*70}")
    t_opt_start = time.time()
    study = optuna.create_study(
        direction='maximize',
        study_name=f'WESAD_Ablation_{variant_name}',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5)
    )
    study.optimize(lambda trial: optuna_objective_ablation(trial, variant_name),
                   n_trials=n_trials, show_progress_bar=True)
    t_opt_dur = time.time() - t_opt_start
    print(f"\n[OPTUNA] Completed {n_trials} trials in {t_opt_dur/60:.2f} mins.")
    print(f"[OPTUNA] Best F1 for {variant_name}: {study.best_value:.4f}")

    best_p = {**study.best_params, 'epochs': 50, 'patience': 10}
    with open(cfg_file, 'w') as f:
        json.dump(best_p, f, indent=2)
    return best_p

In [ ]:
# ==== BLOCK 8: STANDARDIZED 15-FOLD LOSO EVALUATION PIPELINE (WITH TIMING & LIVE PARTIAL OUTPUT) ====

all_sids = list(subject_data_norm.keys())
ablation_results_all = {}

def run_loso_experiment(variant_name, cfg, verbose=True):
    '''Runs 15-fold LOSO evaluation under identical conditions with timing and live partial output.'''
    partial_csv = os.path.join(PRECOMPUTED_DIR,f'loso_results_{variant_name}_partial.csv')
    completed_subjects = []
    fold_results = []
    if os.path.isfile(partial_csv):
      previous_results = pd.read_csv(partial_csv)
      fold_results = previous_results.to_dict('records')
      if 'test_subject' in previous_results.columns:
        completed_subjects = (previous_results['test_subject'].astype(str).tolist())
        print(f"[RESUME] Found {len(completed_subjects)} completed folds")


    best_cfg = {**cfg, 'epochs': cfg.get('epochs', 50), 'patience': cfg.get('patience', 10)}
    best_cfg.setdefault('wd', 5e-4)
    best_cfg.setdefault('label_smoothing', 0.05)
    best_cfg.setdefault('feat_hidden', 32)
    best_cfg.setdefault('tcn_filters', 32)
    best_cfg.setdefault('n_layers', 2)
    best_cfg.setdefault('n_heads', 2)
    best_cfg.setdefault('d_ff', 128)
    best_cfg.setdefault('seq_len_out', 64)
    best_cfg.setdefault('dropout', 0.4)
    best_cfg.setdefault('lr', 3e-4)
    best_cfg.setdefault('focal_gamma', 2.0)
    best_cfg.setdefault('batch_size', 64)

    print(f"\n{'='*85}")
    print(f"RUNNING 15-FOLD LOSO FOR VARIANT: {variant_name.upper()}")
    print(f"Hyperparameters: {best_cfg}")
    print(f"Save Path: {PRECOMPUTED_DIR}")
    print(f"{'='*85}\n")

    t_experiment_start = time.time()
    all_test_probs, all_test_labels, all_test_thresholds = [], [], []

    for fold_i, test_sid in enumerate(all_sids):
        if test_sid in completed_subjects:

          print(f"[SKIP] {test_sid} already completed")

          continue
        t_fold_start = time.time()

        splits = get_loso_splits_v4(subject_data_norm,test_sid) 
        trXr, trXf, trY, vlXr, vlXf, vlY, tsXr, tsXf, tsY, val_sid = splits

        bs = best_cfg.get('batch_size', 64)
        tr_ds = WESADDatasetV4(trXr, trXf, trY, augment=True)
        vl_ds = WESADDatasetV4(vlXr, vlXf, vlY)
        ts_ds = WESADDatasetV4(tsXr, tsXf, tsY)
        tr_loader = DataLoader(tr_ds, batch_size=bs, shuffle=True, drop_last=True)
        vl_loader = DataLoader(vl_ds, batch_size=bs)
        ts_loader = DataLoader(ts_ds, batch_size=bs)

        model = build_ablation_model(
            variant_name, in_channels=NUM_CHANNELS, n_features=N_FEATURES, **best_cfg
        ).to(device)

        model, _ = train_ablation_model(model, tr_loader, vl_loader, best_cfg, verbose=False)

        crit = FocalLoss(gamma=best_cfg.get('focal_gamma', 2.0),
                         label_smoothing=best_cfg.get('label_smoothing', 0.05))

        # Threshold optimization on validation fold
        vl_m = evaluate_ablation(model, vl_loader, crit)
        opt_threshold, val_bal_acc = find_optimal_threshold(vl_m['probs'], vl_m['labels'])

        # -------------------------------------------------
        # Pure inference time (forward pass only)
        # -------------------------------------------------

        model.eval()
        n_test_samples = len(ts_ds)
        with torch.no_grad():
            t_inf_start = time.time()
            for xr, xf, _ in ts_loader:
                xr = xr.to(device)
                if xf is not None:
                    xf = xf.to(device)
                _ = model(xr, xf)
            inference_time = time.time() - t_inf_start
        inference_time_ms_per_sample = (inference_time / n_test_samples) * 1000

        # Evaluation on test fold with optimal threshold
        ts_m = evaluate_ablation(model, ts_loader, crit)
        ts_preds = (ts_m['probs'] >= opt_threshold).astype(int)
        ts_labels = ts_m['labels'].astype(int)

        tp = int(np.sum((ts_preds == 1) & (ts_labels == 1)))
        tn = int(np.sum((ts_preds == 0) & (ts_labels == 0)))
        fp = int(np.sum((ts_preds == 1) & (ts_labels == 0)))
        fn = int(np.sum((ts_preds == 0) & (ts_labels == 1)))
        sens = tp / max(tp + fn, 1)
        spec = tn / max(tn + fp, 1)
        acc = (tp + tn) / max(tp + tn + fp + fn, 1)
        f1 = 2 * tp / max(2 * tp + fp + fn, 1)
        bal_acc = (sens + spec) / 2.0
        fold_time = time.time() - t_fold_start

        fold_record = {
            'fold': fold_i + 1, 'test_subject': test_sid, 'val_subject': val_sid,
            'threshold': round(opt_threshold, 3),
            'accuracy': round(acc, 4), 'balanced_acc': round(bal_acc, 4),
            'f1': round(f1, 4), 'auc': round(ts_m['auc'], 4),
            'sensitivity': round(sens, 4), 'specificity': round(spec, 4),
            'time_sec': round(fold_time, 2),
            'inference_time_sec': round(inference_time, 4),
            'inference_time_ms_per_sample': round(inference_time_ms_per_sample,4)
        }
        fold_results.append(fold_record)
        all_test_probs.append(ts_m['probs'])
        all_test_labels.append(ts_m['labels'])
        all_test_thresholds.append(opt_threshold)

        # -----------------------------------------------------------------
        # LIVE PARTIAL OUTPUT & PARTIAL DISK SAVING (NO NEED TO WAIT!)
        # -----------------------------------------------------------------
        df_partial = pd.DataFrame(fold_results)
        df_partial.to_csv(partial_csv, index=False)

        cum_bal = df_partial['balanced_acc'].mean()
        cum_f1 = df_partial['f1'].mean()
        cum_auc = df_partial['auc'].mean()

        print(f"Fold {fold_i+1:02d}/15 [Test: {test_sid:>3s}] | BalAcc: {bal_acc:.3f} | F1: {f1:.3f} | AUC: {ts_m['auc']:.3f} | Sens: {sens:.3f} | Spec: {spec:.3f} | Thr: {opt_threshold:.2f} | Time: {fold_time:.1f}s")
        print(f"  --> [Partial Progress ({fold_i+1}/15)]: Mean BalAcc={cum_bal:.3f} | Mean F1={cum_f1:.3f} | Mean AUC={cum_auc:.3f} (Saved to {os.path.basename(partial_csv)})\n")

        del model; torch.cuda.empty_cache()

    total_exp_time = time.time() - t_experiment_start
    df_res = pd.DataFrame(fold_results)

    # Save Final CSV
    final_csv = os.path.join(PRECOMPUTED_DIR, f'loso_results_{variant_name}.csv')
    df_res.to_csv(final_csv, index=False)

    print('\n' + '='*85)
    print(f"FINAL 15-FOLD LOSO SUMMARY FOR {variant_name.upper()}")
    print(f"Total Execution Time: {total_exp_time/60:.2f} minutes ({total_exp_time:.1f} seconds)")
    print('='*85)
    print(df_res.to_string(index=False))
    print('-'*85)
    for col in ['accuracy', 'balanced_acc', 'f1', 'auc', 'sensitivity', 'specificity', 'threshold']:
        print(f"  {col:>14s}: {df_res[col].mean():.3f} +/- {df_res[col].std():.3f}")
    print(f"  {'inf ms/sample':>14s}: "f"{df_res['inference_time_ms_per_sample'].mean():.3f} +/- "f"{df_res['inference_time_ms_per_sample'].std():.3f}")
    print(f"  {'time_sec (mean)':>14s}: {df_res['time_sec'].mean():.2f}s per fold")
    print('='*85 + '\n')

    return {
        'df': df_res,
        'probs': all_test_probs,
        'labels': all_test_labels,
        'thresholds': all_test_thresholds,
        'config': best_cfg,
        'total_time_sec': total_exp_time
    }

In [ ]:
# ==== CELL 9: [PROPOSED FULL MODEL] OPTUNA / PREVIOUS CONFIG & 15-FOLD LOSO ====
# Architecture: Multi-Scale TCN + Transformer Encoder + 9-Feature MLP Branch

# Set USE_PREVIOUS_HYPERPARAMS=True to take hyperparams instantly from previous run
# Set RUN_OPTUNA=True only if you wish to re-run fresh 25 Optuna trials
full_params = get_variant_hyperparameters('full', use_previous=True, run_optuna=False)

# Run full 15-fold LOSO with live partial outputs & timing
ablation_results_all['full'] = run_loso_experiment('full', full_params)

In [ ]:
# ==== CELL 10: [VARIANT (i) — WITHOUT FEATURE BRANCH] OPTUNA / PREVIOUS CONFIG & 15-FOLD LOSO ====
# Architecture: Multi-Scale TCN + Transformer Encoder Only (No Handcrafted Feature Branch)

no_features_params = get_variant_hyperparameters('no_features', use_previous=True, run_optuna=False)

# Run full 15-fold LOSO with live partial outputs & timing
ablation_results_all['no_features'] = run_loso_experiment('no_features', no_features_params)

In [ ]:
# ==== CELL 11: [VARIANT (ii) — FEATURE BRANCH ALONE] OPTUNA / PREVIOUS CONFIG & 15-FOLD LOSO ====
# Architecture: 9 Handcrafted Features + MLP Only (No Raw Signal Deep Learning Branch)

feature_only_params = get_variant_hyperparameters('feature_only', use_previous=True, run_optuna=False)

# Run full 15-fold LOSO with live partial outputs & timing
ablation_results_all['feature_only'] = run_loso_experiment('feature_only', feature_only_params)

In [ ]:
# ==== CELL 12: [VARIANT (iii) — WITHOUT TCN] OPTUNA / PREVIOUS CONFIG & 15-FOLD LOSO ====
# Architecture: Direct Linear/Conv Projection + Transformer Encoder + Feature Branch (No Multi-Scale TCN)

no_tcn_params = get_variant_hyperparameters('no_tcn', use_previous=True, run_optuna=False)

# Run full 15-fold LOSO with live partial outputs & timing
ablation_results_all['no_tcn'] = run_loso_experiment('no_tcn', no_tcn_params)

In [ ]:
# ==== CELL 13: [VARIANT (iv) — WITHOUT TRANSFORMER] OPTUNA / PREVIOUS CONFIG & 15-FOLD LOSO ====
# Architecture: Multi-Scale TCN + Global Temporal Pooling + Feature Branch (No Self-Attention Transformer)

no_transformer_params = get_variant_hyperparameters('no_transformer', use_previous=True, run_optuna=False)

# Run full 15-fold LOSO with live partial outputs & timing
ablation_results_all['no_transformer'] = run_loso_experiment('no_transformer', no_transformer_params)

In [ ]:
# ==== CELL 14: [VARIANT (v) — SINGLE-DILATION TCN] OPTUNA / PREVIOUS CONFIG & 15-FOLD LOSO ====
# Architecture: Single-Dilation TCN (d=1) + Transformer Encoder + Feature Branch (No Multi-Scale Dilation Rates)

single_dilation_params = get_variant_hyperparameters('single_dilation_tcn', use_previous=True, run_optuna=False)

# Run full 15-fold LOSO with live partial outputs & timing
ablation_results_all['single_dilation_tcn'] = run_loso_experiment('single_dilation_tcn', single_dilation_params)

In [ ]:
# ==== CELL 15: [ALL-IN-ONE AUTOMATED BATCH RUNNER (OPTIONAL)] ====
# Use this cell if you want to execute all 6 ablation experiments sequentially in batch mode.

RUN_ALL_BATCH = False  # Set to True to trigger full ablation suite automatically

if RUN_ALL_BATCH:
    t_batch_start = time.time()
    for v_name in ['full', 'no_features', 'feature_only', 'no_tcn', 'no_transformer', 'single_dilation_tcn']:
        print(f"\n\n{'#'*85}\n# EXECUTING ABLATION SUITE: {v_name.upper()}\n{'#'*85}")
        best_p = get_variant_hyperparameters(v_name, use_previous=True, run_optuna=False)
        ablation_results_all[v_name] = run_loso_experiment(v_name, best_p)
    print(f"\n[BATCH COMPLETE] Entire ablation suite finished in {(time.time()-t_batch_start)/60:.2f} minutes!")

In [ ]:
# ==== CELL 16: [ABLATION COMPARATIVE ANALYSIS, STATISTICAL TESTS, TIMING & FIGURES] ====
# Generates Master Comparison Table, Paired Wilcoxon/t-tests vs Full Model, Runtime Analysis, and Visualizations.

VARIANT_LABELS = {
    'full':                'Full Hybrid MST-Conformer (Proposed)',
    'no_features':        'Variant (i): No Feature Branch (Raw MS-TCN + Transf.)',
    'feature_only':       'Variant (ii): Feature Branch Alone (9 Feats + MLP)',
    'no_tcn':             'Variant (iii): No TCN (Transformer + Feat. Branch)',
    'no_transformer':     'Variant (iv): No Transformer (MS-TCN + Feat. Branch)',
    'single_dilation_tcn':'Variant (v): Single-Dilation TCN (d=1 + Transf. + Feat.)',
}

# 1. Compile Master Comparison Table (with Execution Times)
rows = []
for var_key, label in VARIANT_LABELS.items():
    if var_key in ablation_results_all:
        df_v = ablation_results_all[var_key]['df']
        tot_time = ablation_results_all[var_key].get('total_time_sec', df_v.get('time_sec', pd.Series([0])).sum())
        row = {
            'Variant': label,
            'Balanced Acc': f"{df_v['balanced_acc'].mean():.3f} ± {df_v['balanced_acc'].std():.3f}",
            'F1-Score': f"{df_v['f1'].mean():.3f} ± {df_v['f1'].std():.3f}",
            'ROC-AUC': f"{df_v['auc'].mean():.3f} ± {df_v['auc'].std():.3f}",
            'Accuracy': f"{df_v['accuracy'].mean():.3f} ± {df_v['accuracy'].std():.3f}",
            'Sensitivity': f"{df_v['sensitivity'].mean():.3f} ± {df_v['sensitivity'].std():.3f}",
            'Specificity': f"{df_v['specificity'].mean():.3f} ± {df_v['specificity'].std():.3f}",
            'Runtime (min)': f"{tot_time/60:.2f} min"
        }
        rows.append(row)

df_comparison = pd.DataFrame(rows)
print("\n" + "="*115)
print("MASTER ABLATION STUDY COMPARISON TABLE (15-Subject LOSO Cross-Validation)")
print("="*115)
print(df_comparison.to_string(index=False))
print("="*115 + "\n")

# 2. Formal Statistical Significance Testing vs. Proposed Full Model
if 'full' in ablation_results_all and len(ablation_results_all) > 1:
    print("="*105)
    print("STATISTICAL SIGNIFICANCE TESTS (Variant vs. Full Model across 15 LOSO Folds)")
    print("="*105)
    df_full = ablation_results_all['full']['df']
    stat_rows = []

    for var_key, label in VARIANT_LABELS.items():
        if var_key == 'full' or var_key not in ablation_results_all:
            continue
        df_v = ablation_results_all[var_key]['df']
        diff_bal_acc = df_full['balanced_acc'].values - df_v['balanced_acc'].values
        diff_f1 = df_full['f1'].values - df_v['f1'].values
        diff_auc = df_full['auc'].values - df_v['auc'].values

        # Paired t-test and Wilcoxon signed rank test
        tt_res_bal = sp_stats.ttest_rel(df_full['balanced_acc'], df_v['balanced_acc'])
        w_res_bal = sp_stats.wilcoxon(df_full['balanced_acc'], df_v['balanced_acc'])

        tt_res_f1 = sp_stats.ttest_rel(df_full['f1'], df_v['f1'])
        w_res_f1 = sp_stats.wilcoxon(df_full['f1'], df_v['f1'])

        # Cohen's d effect size
        cohen_d_bal = np.mean(diff_bal_acc) / (np.std(diff_bal_acc, ddof=1) + 1e-8)
        cohen_d_f1 = np.mean(diff_f1) / (np.std(diff_f1, ddof=1) + 1e-8)

        stat_rows.append({
            'Ablation Variant': var_key,
            'Delta BalAcc': f"{np.mean(diff_bal_acc):+.3f}",
            'p-val (t-test)': f"{tt_res_bal.pvalue:.4f}",
            'p-val (Wilcoxon)': f"{w_res_bal.pvalue:.4f}",
            'Cohen d (BalAcc)': f"{cohen_d_bal:.3f}",
            'Delta F1': f"{np.mean(diff_f1):+.3f}",
            'p-val F1 (Wilcoxon)': f"{w_res_f1.pvalue:.4f}",
            'Cohen d (F1)': f"{cohen_d_f1:.3f}"
        })
    df_stats = pd.DataFrame(stat_rows)
    print(df_stats.to_string(index=False))
    print("="*105 + "\n")
    df_stats.to_csv(os.path.join(PRECOMPUTED_DIR, 'ablation_statistical_tests.csv'), index=False)

# 3. Publication-Ready Visualizations
if len(ablation_results_all) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    keys = [k for k in VARIANT_LABELS.keys() if k in ablation_results_all]
    labels_short = [k.replace('_', ' ').title() for k in keys]
    colors = ['#2E7D32', '#1565C0', '#6A1B9A', '#D84315', '#00838F', '#C2185B'][:len(keys)]

    # 1. Balanced Accuracy Bar Plot
    means_bal = [ablation_results_all[k]['df']['balanced_acc'].mean() for k in keys]
    stds_bal = [ablation_results_all[k]['df']['balanced_acc'].std() for k in keys]
    axes[0, 0].bar(labels_short, means_bal, yerr=stds_bal, capsize=5, color=colors, alpha=0.85, edgecolor='black')
    axes[0, 0].set_ylabel('Balanced Accuracy', fontsize=12)
    axes[0, 0].set_title('(A) Balanced Accuracy across Ablation Variants', fontsize=13, fontweight='bold')
    axes[0, 0].set_ylim(0.4, 0.9)
    axes[0, 0].tick_params(axis='x', rotation=30)
    axes[0, 0].grid(True, axis='y', alpha=0.3)

    # 2. F1-Score Bar Plot
    means_f1 = [ablation_results_all[k]['df']['f1'].mean() for k in keys]
    stds_f1 = [ablation_results_all[k]['df']['f1'].std() for k in keys]
    axes[0, 1].bar(labels_short, means_f1, yerr=stds_f1, capsize=5, color=colors, alpha=0.85, edgecolor='black')
    axes[0, 1].set_ylabel('F1-Score', fontsize=12)
    axes[0, 1].set_title('(B) F1-Score across Ablation Variants', fontsize=13, fontweight='bold')
    axes[0, 1].set_ylim(0.4, 0.9)
    axes[0, 1].tick_params(axis='x', rotation=30)
    axes[0, 1].grid(True, axis='y', alpha=0.3)

    # 3. Aggregated ROC Curves Comparison
    for k, color in zip(keys, colors):
        all_p = np.concatenate(ablation_results_all[k]['probs'])
        all_l = np.concatenate(ablation_results_all[k]['labels'])
        fpr, tpr, _ = roc_curve(all_l, all_p)
        roc_val = auc(fpr, tpr)
        axes[1, 0].plot(fpr, tpr, lw=2, color=color, label=f"{k} (AUC={roc_val:.3f})")
    axes[1, 0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
    axes[1, 0].set_xlabel('False Positive Rate', fontsize=12)
    axes[1, 0].set_ylabel('True Positive Rate', fontsize=12)
    axes[1, 0].set_title('(C) ROC Curves Comparison', fontsize=13, fontweight='bold')
    axes[1, 0].legend(loc='lower right', fontsize=10)
    axes[1, 0].grid(True, alpha=0.3)

    # 4. Sensitivity vs Specificity
    x_indices = np.arange(len(keys))
    w = 0.35
    sens_means = [ablation_results_all[k]['df']['sensitivity'].mean() for k in keys]
    spec_means = [ablation_results_all[k]['df']['specificity'].mean() for k in keys]
    axes[1, 1].bar(x_indices - w/2, sens_means, w, label='Sensitivity', color='#1976D2', alpha=0.85, edgecolor='black')
    axes[1, 1].bar(x_indices + w/2, spec_means, w, label='Specificity', color='#E64A19', alpha=0.85, edgecolor='black')
    axes[1, 1].set_xticks(x_indices)
    axes[1, 1].set_xticklabels(labels_short, rotation=30, ha='right')
    axes[1, 1].set_ylabel('Score', fontsize=12)
    axes[1, 1].set_title('(D) Sensitivity vs Specificity', fontsize=13, fontweight='bold')
    axes[1, 1].set_ylim(0.4, 1.0)
    axes[1, 1].legend(fontsize=11)
    axes[1, 1].grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    fig_path = os.path.join(PRECOMPUTED_DIR, 'ablation_study_comparison.png')
    plt.savefig(fig_path, dpi=200, bbox_inches='tight')
    plt.savefig('ablation_study_comparison.png', dpi=200, bbox_inches='tight')
    plt.show()

# 4. Export Results to Google Drive
if len(rows) > 0:
    df_comparison.to_csv(os.path.join(PRECOMPUTED_DIR, 'ablation_summary_table.csv'), index=False)
for var_key in ablation_results_all:
    ablation_results_all[var_key]['df'].to_csv(os.path.join(PRECOMPUTED_DIR, f'loso_results_{var_key}.csv'), index=False)

print(f"\nAll Ablation v2 results, partial tables, and figures successfully saved to {PRECOMPUTED_DIR}/")